# DiCarlo CKA alignment: result figures

Reads the outputs of one `train_alignment.py` run and its `evaluate_alignment.py` evaluation, and draws five figures. Files are only read, never modified.

**The three models**
- **target**: Tiannan's trained DyRCNNx4 (frozen). It provides the pseudo-labels and the IT-layer representations.
- **aligned**: student trained with CE(pseudo-label) + log(1 − CKA), aligning its IT layer to the target's at the last timestep.
- **control**: same initialization and same data as aligned, trained with CE only (no CKA term).

**Two data sources**
- Figure 1 comes from training (`metrics/*_history.csv`); "val" is the 20% held out from the official train folder.
- Figures 2–5 come from the evaluation (`eval/<name>/`) and are all computed on the **test set** (the official val folder, 3925 images, never seen by the target either).

Timesteps are numbered from 0 (0…19), as in the code.

Only edit `RUN_DIR` (and optionally `EVAL_NAME`) in the first code cell, then run all cells top to bottom.

In [ ]:
RUN_DIR = "/scratch/wz2723/dynvision/runs/smoke_01"   # training run folder
EVAL_NAME = None   # subfolder of eval/; None = the most recent one

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator
import numpy as np
import pandas as pd

run = Path(RUN_DIR)
eval_root = run / "eval"
if EVAL_NAME is None:
    eval_dir = max((p for p in eval_root.iterdir() if p.is_dir()), key=lambda p: p.stat().st_mtime)
else:
    eval_dir = eval_root / EVAL_NAME

hist = {m: pd.read_csv(run / "metrics" / f"{m}_history.csv") for m in ("aligned", "control")}
over_time = pd.read_csv(eval_dir / "metrics_over_time.csv")
cka = pd.read_csv(eval_dir / "cka_over_time.csv")
traj_path = eval_dir / "training_trajectory.csv"
traj = pd.read_csv(traj_path) if traj_path.exists() else None
summary = json.loads((eval_dir / "summary.json").read_text())

try:
    import yaml
    cfg = yaml.safe_load((run / "resolved_config.yaml").read_text())
    CE_FIRST_T = cfg["alignment"]["ce_first_timestep"]
except Exception:
    CE_FIRST_T = None
ALIGN_T = summary["alignment_timestep"]

COLORS = {"target": "#52514e", "aligned": "#2a78d6", "control": "#eb6834", "aligned_vs_control": "#8a8984"}
STYLE = {"target": "--", "aligned": "-", "control": "-"}
plt.rcParams.update({
    "figure.dpi": 110, "axes.grid": True, "grid.color": "#e4e3df", "grid.linewidth": 0.6,
    "axes.spines.top": False, "axes.spines.right": False, "axes.edgecolor": "#8a8984",
    "lines.linewidth": 1.5, "lines.markersize": 3.5, "legend.frameon": False,
    "axes.titlesize": 11, "axes.labelsize": 10, "xtick.labelsize": 9, "ytick.labelsize": 9,
    "legend.fontsize": 9,
})
from matplotlib.lines import Line2D

def marker_for(n_points):
    # Point markers only when there are few points; dense curves stay plain lines.
    return "o" if n_points <= 40 else None

def shared_legend(fig, handles, ncol=None):
    # One legend above all panels, so it never covers data.
    fig.legend(handles=handles, loc="upper center", ncol=ncol or len(handles),
               bbox_to_anchor=(0.5, 1.0))
    fig.tight_layout(rect=(0, 0, 1, 0.92))

def key(label, color, ls="-", marker=None):
    return Line2D([], [], color=color, linestyle=ls, marker=marker, label=label)

print("run:  ", run)
print("eval: ", eval_dir.name)
print(f"epochs trained: {len(hist['aligned'])} | alignment timestep: {ALIGN_T} | CE from timestep: {CE_FIRST_T}")

## Figure 1 — Training curves (train / val during training)

**What is plotted**: x-axis = epoch (0-based); each value is the average over that epoch. Blue = aligned, orange = control. Four panels (2 × 2):

1. **CE loss** (vs pseudo-labels): solid = val, dashed = train. Train CE can sit above val CE because the train loader uses random augmentation and val does not. How well each student learns the target's labels, and whether train and val drift apart (overfitting).
2. **val batch CKA** (student IT vs target IT, last timestep): CKA computed per batch, then averaged over batches. The aligned model optimizes this quantity directly and the control does not, so the gap between the two lines is the effect of the CKA loss.
   - Note: this is a **training monitor**, not a result. With only 48 images per batch, CKA is inflated (n ≪ p). The actual CKA results are Figures 3 and 4 (full test set).
3. **val agreement with target**: fraction of images where the student's last-timestep prediction equals the target's.
4. **val accuracy (ground truth)**: accuracy against the true Imagenette labels. Students are trained on the target's pseudo-labels, not on true labels, so their ceiling is roughly the target's own accuracy.

**How to read it**: curves still clearly rising at the end → training too short, more epochs would help; val getting worse while train keeps improving → overfitting, and the last checkpoint is not the best one.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 7.5))
panels = [
    ("ce_loss", "CE loss vs pseudo-label (lower = better)", True),
    ("batch_cka", "val batch CKA, IT vs target (monitor only)", False),
    ("target_agreement", "val agreement with target", False),
    ("gt_accuracy", "val accuracy (ground truth)", False),
]
mk = marker_for(len(hist["aligned"]))
for ax, (col, title, with_train) in zip(axes.flat, panels):
    for m, df in hist.items():
        ax.plot(df["epoch"], df[f"val_{col}"], "-", color=COLORS[m], marker=mk)
        if with_train:
            ax.plot(df["epoch"], df[f"train_{col}"], "--", color=COLORS[m], alpha=0.7, marker=mk)
    ax.set_title(title)
    ax.set_xlabel("epoch")
    ax.xaxis.set_major_locator(MaxNLocator(integer=True))
shared_legend(fig, [key("aligned", COLORS["aligned"]), key("control", COLORS["control"]),
                    key("val", "#52514e", "-"), key("train (CE panel only)", "#52514e", "--")])
plt.show()

## Figure 2 — Accuracy across timesteps (test set)

**What is plotted**: x-axis = timestep (0…19), i.e. which step of the recurrent network's processing of the same static image; each point is computed over all 3925 test images at that step.

- **Left: accuracy vs ground truth.** Three lines: target (gray dashed), aligned (blue), control (orange). Gray horizontal dashed line = 10% chance level (10 classes).
- **Right: agreement with the target.** Fraction of images where the student's prediction at that step equals the target's pseudo-label (the target's prediction at the last step).

**Two markers**
- Light gray shading = early timesteps that received **no** CE supervision during training (0 to `ce_first_timestep − 1`, the same reaction-time setting the target was trained with). Low accuracy there is expected.
- Vertical dotted line = the timestep used for CKA alignment (the last one).

**How to read it**: the target's curve shows how its recognition builds up over time; the closer a student's curve is to it, the more the student shares the target's temporal dynamics. The key comparison is aligned vs control: CKA only constrains the IT representation at the last step, so check whether it also changes behavior at other timesteps.

In [ ]:
def mark_time_axes(ax):
    if CE_FIRST_T:
        ax.axvspan(-0.5, CE_FIRST_T - 0.5, color="#f0efeb", zorder=0)
    ax.axvline(ALIGN_T, color="#8a8984", linestyle=":", linewidth=1.2)
    ax.set_xlabel("timestep")
    ax.set_xticks(range(0, int(over_time["timestep"].max()) + 1, 2))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.3))
for m, z in (("aligned", 2), ("control", 2), ("target", 3)):   # target drawn on top
    d = over_time[over_time["model"] == m]
    ax1.plot(d["timestep"], d["accuracy"], STYLE[m], color=COLORS[m], marker="o", zorder=z)
ax1.axhline(0.1, color="#b5b4af", linestyle="--", linewidth=1)
ax1.set_title("test accuracy (ground truth)")
ax1.set_ylim(0, 1)
mark_time_axes(ax1)

for m in ("aligned", "control"):
    d = over_time[over_time["model"] == m]
    ax2.plot(d["timestep"], d["agreement_with_pseudolabel"], "-", color=COLORS[m], marker="o")
ax2.set_title("agreement with target's pseudo-label")
ax2.set_ylim(0, 1)
mark_time_axes(ax2)
shared_legend(fig, [key("target", COLORS["target"], "--", "o"),
                    key("aligned", COLORS["aligned"], "-", "o"),
                    key("control", COLORS["control"], "-", "o"),
                    key("chance (10%)", "#b5b4af", "--")])
plt.show()

## Figure 3 — IT-layer CKA across timesteps (test set): the main result

**What is plotted**: x-axis = timestep. Each point is one CKA computed **at once over all 3925 test images** (not a batch average), comparing two models' IT representations at that timestep. Three lines:

- **aligned vs target** (blue): the student pulled toward the target by the CKA loss.
- **control vs target** (orange): the student that only learned the pseudo-labels; the baseline.
- **aligned vs control** (gray dashed): how similar the two students are to each other; a reference.

**Left and right show the same comparisons with two estimators**
- **Left: biased CKA**, the same formula as in the training loss and in the DiCarlo paper. Drawback: when features are very high-dimensional (about 200,000 numbers per image in IT) relative to the number of images, it is inflated overall, so even unrelated representations can score high.
- **Right: unbiased CKA** (Song et al. 2012 estimator), which removes this inflation; unrelated representations score near 0. Values are generally lower than on the left, but the **gap** between aligned and control is more trustworthy.

Vertical dotted line = the timestep aligned during training.

**How to read it**: the key question is whether the blue line is above the orange line, and by how much. The gap should be largest at the aligned timestep (dotted line); a gap at other timesteps means the alignment spread to timesteps that were not directly constrained.

In [ ]:
pairs = [("aligned_vs_target", COLORS["aligned"], "-"),
         ("control_vs_target", COLORS["control"], "-"),
         ("aligned_vs_control", COLORS["aligned_vs_control"], "--")]
fig, axes = plt.subplots(1, 2, figsize=(12, 4.3), sharey=True)
for ax, col, title in zip(axes, ("cka_biased", "cka_unbiased"),
                          ("biased CKA (as in training loss)", "unbiased CKA")):
    for pair, color, ls in pairs:
        d = cka[cka["pair"] == pair]
        ax.plot(d["timestep"], d[col], ls, color=color, marker="o")
    ax.set_title(title)
    mark_time_axes(ax)
axes[0].set_ylabel("CKA (full test set)")
axes[1].tick_params(labelleft=True)
shared_legend(fig, [key(p.replace("_", " "), c, ls, "o") for p, c, ls in pairs])
plt.show()

## Figure 4 — Training trajectory: every saved checkpoint, evaluated on the test set

**What is plotted**: x-axis = epoch; each point is one saved checkpoint (epoch −1 = initialization before training; then one every N epochs, plus the last one). Every checkpoint is re-evaluated on the full test set. Blue = aligned, orange = control.

1. **biased CKA vs target** at the aligned timestep
2. **unbiased CKA vs target** at the same timestep
3. **test accuracy** at the last timestep (vs ground truth)

At epoch −1 both students have identical values (same initialization), so both lines start from the same point (the orange point covers the blue one).

**How to read it**
- CKA: at which epoch the two lines separate, and whether the gap keeps growing → how the alignment effect accumulates with training.
- Accuracy: whether the aligned model pays for its higher CKA with lower recognition performance (blue clearly below orange).
- Whether the curves have flattened over the last few points → whether the number of epochs was enough.

(The smoke test has only 1 epoch, so only two or three points appear here.)

In [ ]:
if traj is None:
    print("No training_trajectory.csv (evaluation was run with --skip_trajectory).")
else:
    fig, axes = plt.subplots(1, 3, figsize=(15, 4.3))
    for ax, col, title in zip(axes,
                              ("cka_biased", "cka_unbiased", "accuracy_final_timestep"),
                              (f"biased CKA vs target (t={ALIGN_T})",
                               f"unbiased CKA vs target (t={ALIGN_T})",
                               "test accuracy (last timestep)")):
        for m in ("aligned", "control"):
            d = traj[traj["model"] == m].sort_values("epoch")
            ax.plot(d["epoch"], d[col], "-", color=COLORS[m], marker="o")
        ax.set_title(title)
        ax.set_xlabel("epoch  (−1 = initialization)")
        ax.xaxis.set_major_locator(MaxNLocator(integer=True))
    shared_legend(fig, [key("aligned", COLORS["aligned"], "-", "o"),
                        key("control", COLORS["control"], "-", "o")])
    plt.show()

## 5 — Summary table

**What is in the table**: the headline test-set numbers for the evaluated student checkpoints (usually the last ones).

- **accuracy (last t)**: accuracy vs ground truth at the last timestep. The target row doubles as a **pipeline check**: it should be close to the level Tiannan reported (62.2% on her own val split, which used random augmentation; here it is the official val folder without augmentation, so a few points of difference are normal). A large gap would mean the preprocessing or the DynVision version differs from her training, and then none of the other numbers can be trusted.
- **agreement with target (last t)**: fraction of images where the student's last-step prediction equals the target's.
- **CKA vs target**: biased / unbiased CKA at the aligned timestep, over the full test set.

The aligned-vs-control CKA is printed above the table as a reference.

In [ ]:
rows = []
for m in ("target", "aligned", "control"):
    row = {"model": m, "accuracy (last t)": summary["accuracy_final_timestep"][m]}
    row["agreement with target (last t)"] = summary["agreement_with_pseudolabel_final_timestep"].get(m, np.nan)
    c = summary["cka_at_alignment_timestep"].get(f"{m}_vs_target", {})
    row[f"CKA vs target, biased (t={ALIGN_T})"] = c.get("biased", np.nan)
    row[f"CKA vs target, unbiased (t={ALIGN_T})"] = c.get("unbiased", np.nan)
    rows.append(row)
table = pd.DataFrame(rows).set_index("model")

ac = summary["cka_at_alignment_timestep"]["aligned_vs_control"]
print(f"run: {run.name} | student checkpoint: {summary['student_checkpoint']} "
      f"(epoch {summary['student_epoch']}) | test images: {summary['n_test_images']}")
print(f"aligned vs control CKA (t={ALIGN_T}): biased {ac['biased']:.3f} | unbiased {ac['unbiased']:.3f}")
table.style.format("{:.3f}", na_rep="—")